# Models
LLMs are powerful AI tools that can interpret and generate text like humans. They’re versatile enough to write content, translate languages, summarize, and answer questions without needing specialized training for each task.
In addition to text generation, many models support:
 Tool calling - calling external tools (like databases queries or API calls) and use results in their responses.
 Structured output - where the model’s response is constrained to follow a defined format.
 Multimodality - process and return data other than text, such as images, audio, and video.
 Reasoning - models perform multi-step reasoning to arrive at a conclusion.

## Basic usage
Models can be utilized in two ways:
### 1: With agents - 
Models can be dynamically specified when creating an agent.
### 2: Standalone - 
Models can be called directly (outside of the agent loop) for tasks like text generation, classification, or extraction without the need for an agent framework.


## Initialize a model
The easiest way to get started with a standalone model in LangChain is to use init_chat_model to initialize one from a chat model provider of your choice (examples below):

### By init_chat_mmodel

In [37]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")
model = init_chat_model(
    "claude-haiku-4-5",
    temperature=0.1,
    max_tokens=20,
    timeout=30,
    max_retries=3,
)

response = model.invoke("Why do parrots talk?")
response.content

'Parrots talk mainly because:\n\n**Social communication** - In the wild, they use'

In [17]:
response = model.invoke("Why do parrots have colorful feathers?")
print(response.content)

# Why Parrots Have Colorful Feathers

Parrots' bright colors serve several important purposes:

**Mate attraction**
- Vivid plumage


In [19]:
conversation = [
    {
        "role": "system",
        "content": "You are a helpful assistant that translates English to French.",
    },
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."},
]

response = model.invoke(conversation)
print(response.content)  # AIMessage("J'adore créer des applications.")

J'adore créer des applications.


# Stream
Most models can stream their output content while it is being generated. By displaying output progressively, streaming significantly improves user experience, particularly for longer responses.
Calling stream() returns an iterator that yields output chunks as they are produced. You can use a loop to process each chunk in real-time:

### 1: Basic text streaming

In [20]:
for chunk in model.stream("Why do parrots have colorful feathers?"):
    print(chunk.text, end="|", flush=True)

|#| Why| Parrots Have Colorful| Feathers|

Parrots'| bright| colors| serve| several important purposes:

**|Mate| attraction|**|
- Viv|id plumage||

### 2: Stream tool calls, reasoning, and other content

In [21]:
for chunk in model.stream("What color is the sky?"):
    for block in chunk.content_blocks:
        if block["type"] == "reasoning" and (reasoning := block.get("reasoning")):
            print(f"Reasoning: {reasoning}")
        elif block["type"] == "tool_call_chunk":
            print(f"Tool call chunk: {block}")
        elif block["type"] == "text":
            print(block["text"])
        else:
            ...


The sky is
 typically
 **
blue**
 during
 the day,
 caused
 by
 Rayleigh scattering of sun
light in
 the atmosphere.


However, the sky
 can appear
 different
 colors depending on conditions
:



In [22]:
full = None  # None | AIMessageChunk
for chunk in model.stream("What color is the sky?"):
    full = chunk if full is None else full + chunk
    print(full.text)

# The
# The sky
# The sky is
# The sky is typically
# The sky is typically blue
# ...

print(full.content_blocks)
# [{"type": "text", "text": "The sky is typically blue..."}]


The sky is
The sky is typically
The sky is typically **
The sky is typically **blue**
The sky is typically **blue** during
The sky is typically **blue** during the day. This happens
The sky is typically **blue** during the day. This happens because of
The sky is typically **blue** during the day. This happens because of a
The sky is typically **blue** during the day. This happens because of a phenomenon called
The sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where
The sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where shorter blue
The sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where shorter blue wavelengths of
The sky is typically **blue** during the day. This happens because of a phenomenon called Rayleigh scattering, where shorter blue wavelengths of sun
The sky is typically **blue** during the day

# Advanced streaming topics

## 1: Streaming events

In [23]:
async for event in model.astream_events("Hello"):

    if event["event"] == "on_chat_model_start":
        print(f"Input: {event['data']['input']}")

    elif event["event"] == "on_chat_model_stream":
        print(f"Token: {event['data']['chunk'].text}")

    elif event["event"] == "on_chat_model_end":
        print(f"Full message: {event['data']['output'].text}")

    else:
        pass

Input: Hello
Token: 
Token: Hello! 
Token: 👋 How can I help you
Token:  today?
Token: 
Full message: Hello! 👋 How can I help you today?


# Batch
Batching a collection of independent requests to a model can significantly improve performance and reduce costs, as the processing can be done in parallel:


In [25]:
responses = model.batch(
    [
        "Why do parrots have colorful feathers?",
        "How do airplanes fly?",
        "What is quantum computing?",
    ]
)
for response in responses:
    print(response.content)

# Why Parrots Have Colorful Feathers

Parrots' bright colors serve several important purposes:

**Mate attraction**
- Vivid plumage
# How Airplanes Fly

Airplanes stay aloft through **lift**, which is generated by their wings moving through the air. Here's the basic principle:

## The
# Quantum Computing

Quantum computing is a type of computation that harnesses the principles of quantum mechanics to process information in fundamentally different ways than classical computers.

## Key Differences from Classical


In [28]:
for response in model.batch_as_completed(
    [
        "Why do parrots have colorful feathers?",
        "How do airplanes fly?",
        "What is quantum computing?",
    ]
):
    print(response)

(2, AIMessage(content='# Quantum Computing\n\nQuantum computing is a type of computation that harnesses the principles of quantum', additional_kwargs={}, response_metadata={'id': 'msg_01XCSVJnDibRsofpFuL4hAg1', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'max_tokens', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 12, 'output_tokens': 20, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'}, id='lc_run--019cdacf-a6a8-7cc3-81d7-dfcc638fce8d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 20, 'total_tokens': 32, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}))
(1, AIMessage(content

# Tool calling

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
### 1: A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
### 2: A function or coroutine to execute.

In [30]:
from langchain.tools import tool


@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")

print(response.content)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

[]


When binding user-defined tools, the model’s response includes a request to execute a tool. When using a model separately from an agent, it is up to you to execute the requested tool and return the result back to the model for use in subsequent reasoning. When using an agent, the agent loop will handle the tool execution loop for you.
Below, we show some common ways you can use tool calling.

### 1: Tool execution loop

In [36]:
# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
print(ai_msg)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

content=[] additional_kwargs={} response_metadata={'id': 'msg_01MJvLMNoVRumC14BkW272DC', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_reason': 'max_tokens', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 564, 'output_tokens': 20, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--019cdad7-ff33-7ee2-9b66-e2c98226ff37-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 564, 'output_tokens': 20, 'total_tokens': 584, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}

